# Ch.1 — Logistic Regression for Smiling Detection

**FaceAI**: Classify celebrity faces as Smiling/Not-Smiling using logistic regression on HOG features.

**Dataset**: CelebA subset (5,000 images, 64×64 grayscale, HOG descriptors)

**Target**: ~88% accuracy baseline

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import numpy as np, matplotlib.pyplot as plt, Path from pathlib
# 2. Import LogisticRegression from sklearn.linear_model
# 3. Import train_test_split from sklearn.model_selection
# 4. Import StandardScaler from sklearn.preprocessing
# 5. Import classification_report, confusion_matrix, roc_auc_score,
#    roc_curve, ConfusionMatrixDisplay from sklearn.metrics
# 6. Create IMG_DIR = Path("img"); call IMG_DIR.mkdir(exist_ok=True)
# 7. Set SAVE_KW = dict(dpi=150, bbox_inches='tight') and np.random.seed(42)
#
# Hint:
#   from sklearn.metrics import (classification_report, confusion_matrix,
#                                roc_auc_score, roc_curve, ConfusionMatrixDisplay)
#   IMG_DIR = Path("img")
#   IMG_DIR.mkdir(exist_ok=True)
#   np.random.seed(???)


## §0 Data — CelebA Smiling Attribute

We use a synthetic proxy that mimics CelebA's Smiling distribution (48% positive).
Replace with real CelebA via `torchvision.datasets.CelebA` when available.

**CelebA quick-start** (replace synthetic proxy):
```python
# Option 1: Kaggle mirror (jessicali9530/celeba-dataset)
# Option 2: Official CelebA — download aligned images + list_attr_celeba.txt
#
# Folder layout expected:
# data/celeba/img_align_celeba/ — face images (000001.jpg, ...)
# data/celeba/metadata/list_attr_celeba.txt
#
# Minimal loader:
# from pathlib import Path
# import pandas as pd
# attr = pd.read_csv('data/celeba/metadata/list_attr_celeba.txt',
# sep=r'\s+', skiprows=1)
# attr = (attr + 1) // 2 # {-1,+1} → {0,1}
# y_smiling = attr['Smiling'].astype(int)
# # Use official train/val/test splits to avoid leakage
# # Persist scaler/PCA/HOG settings alongside the model
```

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import make_classification from sklearn.datasets
# 2. Generate 5000 samples, 200 features, 40 informative, 20 redundant,
#    3 clusters_per_class, weights=[0.52, 0.48], flip_y=0.05, random_state=42
# 3. Split into train/test using train_test_split() with test_size=0.2,
#    stratify=y, random_state=42
# 4. Print shapes and smiling rate for train and test splits
#
# Hint:
#   from sklearn.datasets import make_classification
#   X, y = make_classification(n_samples=???, n_features=???,
#                              n_informative=???, n_redundant=???,
#                              n_clusters_per_class=???, weights=???,
#                              flip_y=???, random_state=42)
#   X_train, X_test, y_train, y_test = train_test_split(
#       X, y, test_size=???, stratify=???, random_state=42)


## §1 The Sigmoid Function

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create z = np.linspace(-6, 6, 300) for the logit axis
# 2. Compute sigma = 1 / (1 + np.exp(-z))
# 3. Plot z vs sigma; add horizontal dashed line at 0.5 and vertical at 0
# 4. Label axes ('Logit z = w·x + b', 'σ(z) = P(Smiling)') and add title
# 5. Add annotation arrow at the threshold=0.5 crossing point
# 6. Save figure to IMG_DIR / 'sigmoid.png' using SAVE_KW
#
# Hint:
#   z = np.linspace(???, ???, 300)
#   sigma = 1 / (1 + np.exp(???))
#   ax.axhline(???, color='gray', linestyle='--')
#   ax.axvline(???, color='gray', linestyle='--')
#   fig.savefig(IMG_DIR / 'sigmoid.png', **SAVE_KW)


## §2 Training Logistic Regression

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Create StandardScaler(); call fit_transform(X_train) -> X_train_s
#    and transform(X_test) -> X_test_s  (never fit on test data!)
# 2. Create LogisticRegression(C=1.0, max_iter=500, solver='lbfgs', random_state=42)
# 3. Call model.fit(X_train_s, y_train)
# 4. Compute train_acc = model.score(X_train_s, y_train) and
#    test_acc = model.score(X_test_s, y_test); print both
#
# Hint:
#   scaler = StandardScaler()
#   X_train_s = scaler.fit_transform(???)
#   X_test_s  = scaler.transform(???)   # transform only — no re-fitting!
#   model = LogisticRegression(C=???, max_iter=???, solver='lbfgs', random_state=42)
#   model.fit(???, ???)


## §3 Confusion Matrix

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get y_pred = model.predict(X_test_s)
# 2. Create a figure + axes; call ConfusionMatrixDisplay.from_predictions(
#    y_test, y_pred, display_labels=['Not Smiling', 'Smiling'], cmap='Blues', ax=ax)
# 3. Set title 'Confusion Matrix — Smiling Detection' and save to 'confusion_matrix.png'
# 4. Print classification_report(y_test, y_pred, target_names=[...])
#
# Hint:
#   y_pred = model.predict(???)
#   ConfusionMatrixDisplay.from_predictions(
#       y_test, y_pred, display_labels=???, cmap='Blues', ax=ax)
#   print(classification_report(???, ???, target_names=['Not Smiling', 'Smiling']))


## §4 Binary Cross-Entropy Loss

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute sigma = 1 / (1 + np.exp(-z)) for z = np.linspace(-6, 6, 200)
# 2. Panel A: mse_loss = (1 - sigma)**2 for y=1; plot vs z; annotate vanishing
#    gradient regions at both extremes of z
# 3. Panel B: bce_loss = -np.log(sigma) for y=1; plot vs z; annotate the
#    large penalty for confident wrong predictions
# 4. Use fig, axes = plt.subplots(1, 2, figsize=(13, 5)) with supertitle
# 5. Save to IMG_DIR / 'mse_vs_bce_loss.png' and print key difference
#
# Hint:
#   sigma    = 1 / (1 + np.exp(-z))
#   mse_loss = (1 - sigma) ** ???       # y=1 case
#   bce_loss = -np.log(???)             # y=1: -log(p_hat)
#   fig, axes = plt.subplots(1, 2, figsize=(13, 5))
#   axes[0].plot(z, mse_loss, 'b-', label='MSE when y=1')
#   axes[1].plot(z, bce_loss, 'r-', label='BCE when y=1')


## §5 ROC Curve

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get y_prob = model.predict_proba(X_test_s)[:, 1]
# 2. Compute fpr, tpr, thresholds = roc_curve(y_test, y_prob)
# 3. Compute auc = roc_auc_score(y_test, y_prob)
# 4. Plot ROC curve with AUC in the label; add k-- diagonal for random baseline
# 5. Save to IMG_DIR / 'roc_curve.png'
#
# Hint:
#   y_prob = model.predict_proba(???)[:, 1]
#   fpr, tpr, thresholds = roc_curve(???, ???)
#   auc = roc_auc_score(???, ???)
#   ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'LogReg (AUC={auc:.3f})')
#   ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')


## §6 Probability Distribution

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Plot overlapping histograms of y_prob split by true class
#    (y_prob[y_test==0] in blue, y_prob[y_test==1] in orange)
# 2. Use bins=30, alpha=0.6 for each histogram
# 3. Add vertical dashed line at threshold 0.5 in red
# 4. Label axes 'Predicted P(Smiling)' and 'Count'; add legend
# 5. Save to IMG_DIR / 'prob_distribution.png'
#
# Hint:
#   ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6,
#           label='Not Smiling', color='blue')
#   ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6,
#           label='Smiling', color='orange')
#   ax.axvline(???, color='red', linestyle='--', label='Threshold=0.5')


## §7 Feature Importance (Top Weights)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Get coefs = model.coef_[0]  (shape: n_features,)
# 2. Select top_k=20 indices by absolute value:
#    np.argsort(np.abs(coefs))[-top_k:][::-1]
# 3. Assign color: 'green' where coef > 0, 'red' where coef < 0
# 4. Plot horizontal bar chart (barh); label y-ticks as 'HOG[i]'
# 5. Call ax.invert_yaxis(); save to IMG_DIR / 'feature_weights.png'
#
# Hint:
#   coefs   = model.coef_[0]
#   top_idx = np.argsort(np.abs(coefs))[-???:][::-1]
#   colors  = ['green' if c > 0 else 'red' for c in coefs[top_idx]]
#   ax.barh(range(top_k), coefs[top_idx], color=colors)
#   ax.invert_yaxis()


## §8 Threshold Sweep

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Import f1_score, precision_score, recall_score from sklearn.metrics
# 2. Define thresholds_sweep = np.arange(0.1, 0.9, 0.02)
# 3. Loop: for each t binarize y_prob → (y_prob >= t).astype(int);
#    compute and store F1, precision, recall
# 4. Plot all three curves vs threshold; mark best_t = argmax(F1) with axvline
# 5. Save to IMG_DIR / 'threshold_sweep.png'
#
# Hint:
#   from sklearn.metrics import f1_score, precision_score, recall_score
#   thresholds_sweep = np.arange(???, ???, 0.02)
#   for t in thresholds_sweep:
#       y_t = (y_prob >= t).astype(int)
#       f1s.append(f1_score(???, ???))
#   best_t = thresholds_sweep[np.argmax(f1s)]


## §9 Summary

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Print separator line and chapter title
# 2. Print test_acc, auc, best_t, max(f1s)
# 3. Print ACCURACY constraint status (Ch.1 achieves ~88%, 2% below target)
# 4. Print a note pointing to Ch.4/Ch.5 to break 90%
#
# Hint:
#   print("=" * 50)
#   print("Ch.1 — Logistic Regression for Smiling Detection")
#   print(f"Test Accuracy:  {test_acc:.3f}")
#   print(f"ROC-AUC:        {auc:.3f}")
#   print(f"Best Threshold: {best_t:.2f}")
#   print(f"Best F1:        {max(f1s):.3f}")


## Is This Enough to Cover Logistic Regression?

**Yes — for a solid production baseline.** This notebook covers:

| Concept | Covered | Where |
|---------|---------|-------|
| Sigmoid activation | | §1 |
| Feature scaling | | §2 (`StandardScaler`) |
| Binary cross-entropy loss | | §4 (MSE vs BCE panel) |
| Gradient descent training | | sklearn `lbfgs` solver |
| Confusion matrix | | §3 |
| ROC-AUC | | §5 |
| Threshold tuning | | §8 |

**What this notebook doesn't cover (by design):**
- Multi-class (softmax): → Ch.3 Metrics / Topic 03 Neural Networks
- Regularization deep-dive (L1/L2): → Ch.5 Hyperparameter Tuning
- Full CelebA pipeline (202k images): → replace synthetic proxy in §0
- Probability calibration (Platt scaling): → a production add-on, see sklearn `CalibrationDisplay`

**Is logistic regression enough for FaceAI?** It achieves ~88% on Smiling — 2% below the 90% target. You need Ch.4 (SVM) or Ch.5 (tuning) to break 90%. But logistic regression's speed (<1ms inference) and calibrated probabilities make it the right baseline to beat.

## Exercises

1. **Regularization sweep**: Train LogReg with C ∈ {0.001, 0.01, 0.1, 1, 10, 100}. Plot train/test accuracy vs C.
2. **Multi-attribute**: Train separate LogReg models for Eyeglasses (13%) and Bald (2.5%). Compare accuracy vs F1.
3. **Feature comparison**: Compare raw pixel features (4096-dim) vs the current features. Which gives better AUC?

In [ ]:
# Exercise 1: Regularization sweep
# TODO: Implement this cell
#
# Hint: for C in [0.001, 0.01, 0.1, 1, 10, 100]: train LogisticRegression(C=C),
#   record train/test accuracy; plot accuracy vs log(C) to visualise regularisation


In [ ]:
# Exercise 2: Multi-attribute classification
# TODO: Implement this cell
#
# Hint: make_classification(weights=[0.87, 0.13]) for Eyeglasses and
#   make_classification(weights=[0.975, 0.025]) for Bald;
#   train separate LogisticRegression models; compare accuracy vs F1 for each


In [ ]:
# Exercise 3: Feature comparison
# TODO: Implement this cell
#
# Hint: make_classification(n_features=4096, n_informative=???) for pixel-like features;
#   train LogReg on both 200-feat and 4096-feat data; compare roc_auc_score
